# Reproduction of manuscript Figures 1–3

Three-site calculations for Eq. (32): Fig. 1 (strong/Redfield-like), Fig. 2 (weak/Förster-like), and Fig. 3 (mixed regime).

The calculation uses the QME and basis optimization defined in the accompanying manuscript. Site populations are labeled 1, 2, and 3.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from QME import Config, run_system
from manuscript_models import three_site_model

OUT = Path("results")
OUT.mkdir(exist_ok=True)

# (V12, V23, DeltaE12, DeltaE23, lambda) in cm^-1
cases = {
    1: (100.0, 100.0, 20.0, 20.0, 20.0),
    2: (20.0, 20.0, 100.0, 100.0, 100.0),
    3: (100.0, 20.0, 20.0, 100.0, 50.0),
}

cfg = Config(
    temperature_k=77.0,
    tau_g_ps=0.05,
    nmax=100_000,
    dt_ps=1.0e-4,
    seed=12345,
)


In [ ]:
results = {}
for fig_no, (V12, V23, dE12, dE23, lam) in cases.items():
    H = three_site_model(V12, V23, dE12, dE23)
    lambda_i = np.full(3, lam)

    for rep in ("present", "cmrt", "forster"):
        results[fig_no, rep] = run_system(
            H, lambda_i, cfg, rep, initial_site=0,
            output_dir=OUT, tag=f"Fig{fig_no}_{rep}",
        )

    print(f"Fig. {fig_no}")
    for rep in ("present", "cmrt", "forster"):
        r = results[fig_no, rep]
        print(f"  {rep:7s} Vave = {r['basis']['vave']:.6f} cm^-1")


In [ ]:
PAPER_COLORS_3 = ["#ee2c2a", "#6dbe44", "#2555a6"]  # sites 1, 2, 3

for fig_no in (1, 2, 3):
    fig, axs = plt.subplots(1, 3, figsize=(11.5, 3.4), sharex=True, sharey=True)
    panels = [("present", "(a) Present"), ("cmrt", "(b) CMRT"), ("forster", "(c) Förster")]

    for ax, (rep, title) in zip(axs, panels):
        r = results[fig_no, rep]
        mask = r["time_ps"] > 0
        for i, label in enumerate(("1", "2", "3")):
            ax.plot(
                r["time_ps"][mask], r["populations"][mask, i],
                color=PAPER_COLORS_3[i], linewidth=1.2, label=label,
            )
        ax.set_xscale("log")
        ax.set_xlim(1e-3, 10)
        ax.set_ylim(0, 1)
        ax.set(xlabel="Time (ps)", ylabel="Probability", title=title)
        ax.set_box_aspect(1)
        ax.legend(frameon=False)

    fig.tight_layout()
    fig.savefig(OUT / f"Fig{fig_no}.pdf", bbox_inches="tight")
    plt.show()
